# Indexación y Particionamiento en PostgreSQL

---

## Unidad 4: Optimización en PostgreSQL

**Maestría en Arquitectura de Software**  
**Asignatura:** Diseño y Optimización de Bases de Datos  
**Universidad de La Sabana**

**Integrantes:**     

Henry Julian Salazar Salcedo - 0000396117

Santiago Eduardo Munoz Castillo - 0000394453

Wolfran Alirio Pinzon Murillo - 0000393439

Edward Augusto Ramirez Rodriguez - 0000324316

Equipo E10

---


## Introducción al momento:

Aplicación práctica de técnicas de optimización en la base de datos real de Ecommify
implementada en Supabase.

---

## Objetivos

1. **Implementar** diferentes tipos de índices (B-tree, GIN, GiST, BRIN) según el tipo de consulta y datos
2. **Crear** índices compuestos y parciales para optimizar consultas específicas
3. **Aplicar** particionamiento declarativo RANGE para mejorar el rendimiento en grandes volúmenes de datos
4. **Implementar** vistas materializadas para acelerar consultas analíticas complejas
5. **Medir** y comparar mejoras de rendimiento mediante planes de ejecución



## 1️. Configuración del entorno

### Instalación de librerías necesarias

Se instalarán las bibliotecas necesarias para la conexión con Supabase, análisis de datos y visualización de resultados.

In [ ]:
# Instalación de librerías
!pip install pandas numpy matplotlib seaborn plotly psycopg2-binary sqlalchemy --quiet

print("Librerías instaladas correctamente")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 29.7 MB/s eta 0:00:00
Librerías instaladas correctamente


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from sqlalchemy import create_engine, text
import psycopg2
from psycopg2 import sql, extras
from getpass import getpass

# Configuración de visualización
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Configuración de tamaños de figura por defecto
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Bibliotecas importadas exitosamente")
print(f"Versión de pandas: {pd.__version__}")
print(f"Versión de SQLAlchemy: {__import__('sqlalchemy').__version__}")

Bibliotecas importadas exitosamente
Versión de pandas: 2.2.2
Versión de SQLAlchemy: 2.0.50


## 2️. Conexión a Supabase PostgreSQL

### Configuración de credenciales

Se establece la conexión con la base de datos PostgreSQL en Supabase utilizando credenciales seguras.

In [ ]:
# Solicitar credenciales de forma segura
print("🔐 CONFIGURACIÓN DE CONEXIÓN A SUPABASE")
print("=" * 80)

# Solicitar el connection string completo
connection_string = getpass("Connection String: ")

# Validar que se ingresó un valor
if not connection_string or not connection_string.startswith('postgresql://'):
    print("❌ Error: Connection string inválido. Debe comenzar con 'postgresql://'")
else:
    print("\n Credenciales recibidas")
    print(" Probando conexión...\n")

🔐 CONFIGURACIÓN DE CONEXIÓN A SUPABASE
Connection String: ··········

 Credenciales recibidas
 Probando conexión...



### Probar conexión a Supabase

In [ ]:
# Función de conexión
def get_connection():
    """
    Crea y retorna una conexión a PostgreSQL usando connection string.

    Returns:
        connection: Objeto de conexión psycopg2
    """
    try:
        conn = psycopg2.connect(connection_string)
        return conn
    except Exception as e:
        print(f"Error al conectar: {e}")
        return None

# Verificar conexión
conn = get_connection()
if conn:
    print("Conexión exitosa a Supabase PostgreSQL")
    conn.close()
else:
    print("❌ Error en la conexión")

Conexión exitosa a Supabase PostgreSQL


## 3. Preparación: Creación de tabla de trabajo

### Setup de entorno de pruebas

**Nota sobre organización de schemas:**

Es una buena práctica organizacional crear un schema dedicado para análisis dado los siguientes beneficios:

**Beneficios de usar schemas dedicados:**
- **Organización:** Separar tablas de producción de tablas de análisis/optimización
- **Seguridad:** Facilitar la gestión de permisos por schema
- **Mantenibilidad:** Simplificar limpieza y gestión de objetos relacionados
- **Escalabilidad:** Preparar para arquitecturas multi-tenant o multi-proyecto

por tanto crearemos el schema `ecommify_olist_analytics` para alojar nuestras tablas de optimización, manteniendo así una mejor organización del workspace de desarrollo.

---

Para demostrar las técnicas de indexación y particionamiento, crearemos un schema dedicado y una tabla optimizada con datos del dataset ecommify_olist.

In [ ]:
# Crear schema y tablas para ejercicios de indexación
conn = get_connection()
cur = conn.cursor()

# Crear schema si no existe
cur.execute("""
    CREATE SCHEMA IF NOT EXISTS ecommify_olist_analytics;
""")

conn.commit()
print("Schema ecommify_olist_analytics verificado/creado")

Schema ecommify_olist_analytics verificado/creado


In [ ]:
def execute_sql(sql_query, fetch=False, params=None):
    """Executes an SQL query and returns results if fetch is True."""
    conn = None
    result = None
    try:
        conn = get_connection()
        if conn:
            with conn.cursor(cursor_factory=psycopg2.extras.DictCursor) as cur:
                cur.execute(sql_query, params) # Pass parameters here
                conn.commit()
                if fetch:
                    result = cur.fetchall()
    except Exception as e:
        print(f"❌ Error executing SQL: {e}\nQuery: {sql_query[:200]}...")
    finally:
        if conn:
            conn.close()
    return result

print("Helper function `execute_sql` defined.")

Helper function `execute_sql` defined.


### 3.1 Creación y Carga de Tablas Optimizadas

Se procederá a crear las tablas `customers`, `orders`, `order_items`, `order_payments` y `sellers` en el esquema `ecommify_olist_analytics`. Cada tabla será eliminada si ya existe, creada con la misma estructura que su contraparte en el esquema `public`, y luego poblada con los datos correspondientes.

In [ ]:
tables_to_create = [
    'customers',
    'orders',
    'order_items',
    'order_payments',
    'sellers'
]

for table_name in tables_to_create:
    print(f"\nProcessing table: {table_name}")

    # 1. Drop table if it exists in the analytics schema
    drop_query = sql.SQL("DROP TABLE IF EXISTS ecommify_olist_analytics.{}; ").format(sql.Identifier(table_name))
    execute_sql(drop_query.as_string(get_connection().cursor()))
    print(f"  ✅Eliminar existencia de la tabla ecommify_olist_analytics.{table_name} (if it existed).")

    # 2. Create table with exact structure (including PKs) from public schema
    create_table_like_query = sql.SQL("CREATE TABLE ecommify_olist_analytics.{} (LIKE public.{} INCLUDING ALL); ").format(
        sql.Identifier(table_name),
        sql.Identifier(table_name)
    )
    execute_sql(create_table_like_query.as_string(get_connection().cursor()))
    print(f"  ✅ Crea tabla ecommify_olist_analytics.{table_name} con la misma informacón del schema public.{table_name}.")

    # 3. Copy all data from public table to the new table
    copy_data_query = sql.SQL("INSERT INTO ecommify_olist_analytics.{} SELECT * FROM public.{}; ").format(
        sql.Identifier(table_name),
        sql.Identifier(table_name)
    )
    execute_sql(copy_data_query.as_string(get_connection().cursor()))
    print(f"  ✅ Copia datos desde el schema public.{table_name} to ecommify_olist_analytics.{table_name}.")

print("\nTodos los datos de las tablas fueron exportados al nuevo schema.")


Processing table: customers
  ✅Eliminar existencia de la tabla ecommify_olist_analytics.customers (if it existed).
  ✅ Crea tabla ecommify_olist_analytics.customers con la misma informacón del schema public.customers.
  ✅ Copia datos desde el schema public.customers to ecommify_olist_analytics.customers.

Processing table: orders
  ✅Eliminar existencia de la tabla ecommify_olist_analytics.orders (if it existed).
  ✅ Crea tabla ecommify_olist_analytics.orders con la misma informacón del schema public.orders.
  ✅ Copia datos desde el schema public.orders to ecommify_olist_analytics.orders.

Processing table: order_items
  ✅Eliminar existencia de la tabla ecommify_olist_analytics.order_items (if it existed).
  ✅ Crea tabla ecommify_olist_analytics.order_items con la misma informacón del schema public.order_items.
  ✅ Copia datos desde el schema public.order_items to ecommify_olist_analytics.order_items.

Processing table: order_payments
  ✅Eliminar existencia de la tabla ecommify_olist_a

## 4️. Tipos de índices en PostgreSQL

### Índices especializados para diferentes casos de uso

PostgreSQL ofrece diferentes tipos de índices optimizados para distintos casos de uso:

- **B-tree**: Índice por defecto, ideal para comparaciones de igualdad y rango
- **Hash**: Optimizado para búsquedas de igualdad exacta
- **GIN (Generalized Inverted Index)**: Para búsquedas en arrays, JSONB y texto completo
- **GiST (Generalized Search Tree)**: Para datos geométricos y búsquedas complejas
- **BRIN (Block Range Index)**: Para tablas grandes con datos ordenados naturalmente

### 4.1. Índice B-tree (predeterminado)

Los índices B-tree son efectivos para búsquedas de igualdad, rangos y ordenamiento. Son el tipo predeterminado en PostgreSQL.

A continuación, identificaremos columnas clave en nuestras tablas recién creadas en el esquema `ecommify_olist_analytics` que se beneficiarían de un índice B-tree. Esto incluye columnas utilizadas en cláusulas `WHERE`, `ORDER BY` y en uniones (`JOIN`s), especialmente las claves foráneas que no son parte de una clave primaria.

In [ ]:
print("\n--- Creando índices B-tree en ecommify_olist_analytics ---")

# Indexes for 'customers' table
execute_sql("CREATE INDEX IF NOT EXISTS idx_customers_unique_id ON ecommify_olist_analytics.customers (customer_unique_id);")
print("  ✅ Index 'idx_customers_unique_id' created on ecommify_olist_analytics.customers.")
execute_sql("CREATE INDEX IF NOT EXISTS idx_customers_zip_code_prefix ON ecommify_olist_analytics.customers (customer_zip_code_prefix);")
print("  ✅ Index 'idx_customers_zip_code_prefix' created on ecommify_olist_analytics.customers.")

# Indexes for 'orders' table
execute_sql("CREATE INDEX IF NOT EXISTS idx_orders_customer_id ON ecommify_olist_analytics.orders (customer_id);")
print("  ✅ Index 'idx_orders_customer_id' created on ecommify_olist_analytics.orders (Foreign Key).")
execute_sql("CREATE INDEX IF NOT EXISTS idx_orders_status ON ecommify_olist_analytics.orders (order_status);")
print("  ✅ Index 'idx_orders_status' created on ecommify_olist_analytics.orders.")
execute_sql("CREATE INDEX IF NOT EXISTS idx_orders_purchase_timestamp ON ecommify_olist_analytics.orders (order_purchase_timestamp);")
print("  ✅ Index 'idx_orders_purchase_timestamp' created on ecommify_olist_analytics.orders.")

# Indexes for 'order_items' table
execute_sql("CREATE INDEX IF NOT EXISTS idx_order_items_order_id ON ecommify_olist_analytics.order_items (order_id);")
print("  ✅ Index 'idx_order_items_order_id' created on ecommify_olist_analytics.order_items (Foreign Key).")
execute_sql("CREATE INDEX IF NOT EXISTS idx_order_items_product_id ON ecommify_olist_analytics.order_items (product_id);")
print("  ✅ Index 'idx_order_items_product_id' created on ecommify_olist_analytics.order_items (Foreign Key).")
execute_sql("CREATE INDEX IF NOT EXISTS idx_order_items_seller_id ON ecommify_olist_analytics.order_items (seller_id);")
print("  ✅ Index 'idx_order_items_seller_id' created on ecommify_olist_analytics.order_items (Foreign Key).")

# Indexes for 'order_payments' table
execute_sql("CREATE INDEX IF NOT EXISTS idx_order_payments_order_id ON ecommify_olist_analytics.order_payments (order_id);")
print("  ✅ Index 'idx_order_payments_order_id' created on ecommify_olist_analytics.order_payments (Foreign Key).")
execute_sql("CREATE INDEX IF NOT EXISTS idx_order_payments_type ON ecommify_olist_analytics.order_payments (payment_type);")
print("  ✅ Index 'idx_order_payments_type' created on ecommify_olist_analytics.order_payments.")

# Indexes for 'sellers' table
execute_sql("CREATE INDEX IF NOT EXISTS idx_sellers_zip_code_prefix ON ecommify_olist_analytics.sellers (seller_zip_code_prefix);")
print("  ✅ Index 'idx_sellers_zip_code_prefix' created on ecommify_olist_analytics.sellers.")

print("\n--- Todos los índices B-tree propuestos han sido creados o verificados. ---")


--- Creando índices B-tree en ecommify_olist_analytics ---
  ✅ Index 'idx_customers_unique_id' created on ecommify_olist_analytics.customers.
  ✅ Index 'idx_customers_zip_code_prefix' created on ecommify_olist_analytics.customers.
  ✅ Index 'idx_orders_customer_id' created on ecommify_olist_analytics.orders (Foreign Key).
  ✅ Index 'idx_orders_status' created on ecommify_olist_analytics.orders.
  ✅ Index 'idx_orders_purchase_timestamp' created on ecommify_olist_analytics.orders.
  ✅ Index 'idx_order_items_order_id' created on ecommify_olist_analytics.order_items (Foreign Key).
  ✅ Index 'idx_order_items_product_id' created on ecommify_olist_analytics.order_items (Foreign Key).
  ✅ Index 'idx_order_items_seller_id' created on ecommify_olist_analytics.order_items (Foreign Key).
  ✅ Index 'idx_order_payments_order_id' created on ecommify_olist_analytics.order_payments (Foreign Key).
  ✅ Index 'idx_order_payments_type' created on ecommify_olist_analytics.order_payments.
  ✅ Index 'idx_sel

### 4.2. Índice Hash

Los índices Hash son altamente eficientes para búsquedas de igualdad exacta (`=`). Sin embargo, no soportan búsquedas de rango (`>`, `<`, `BETWEEN`) ni operaciones de ordenamiento (`ORDER BY`). Son útiles para columnas donde solo se realizan búsquedas de valores específicos, como IDs, códigos, o valores categóricos.

Se identificarón algunas columnas en el esquema `ecommify_olist_analytics` que se benefician de un índice Hash por su uso frecuente en búsquedas de igualdad exacta.

In [ ]:
print("\n--- Creando índices Hash en ecommify_olist_analytics ---")

# Indexes for 'customers' table
execute_sql("CREATE INDEX IF NOT EXISTS idx_customers_city_hash ON ecommify_olist_analytics.customers USING HASH (customer_city);")
print("  ✅ Index 'idx_customers_city_hash' created on ecommify_olist_analytics.customers.")
execute_sql("CREATE INDEX IF NOT EXISTS idx_customers_state_hash ON ecommify_olist_analytics.customers USING HASH (customer_state);")
print("  ✅ Index 'idx_customers_state_hash' created on ecommify_olist_analytics.customers.")

# Indexes for 'orders' table
execute_sql("CREATE INDEX IF NOT EXISTS idx_orders_status_hash ON ecommify_olist_analytics.orders USING HASH (order_status);")
print("  ✅ Index 'idx_orders_status_hash' created on ecommify_olist_analytics.orders.")

# Indexes for 'order_payments' table
execute_sql("CREATE INDEX IF NOT EXISTS idx_order_payments_type_hash ON ecommify_olist_analytics.order_payments USING HASH (payment_type);")
print("  ✅ Index 'idx_order_payments_type_hash' created on ecommify_olist_analytics.order_payments.")

# Indexes for 'sellers' table
execute_sql("CREATE INDEX IF NOT EXISTS idx_sellers_city_hash ON ecommify_olist_analytics.sellers USING HASH (seller_city);")
print("  ✅ Index 'idx_sellers_city_hash' created on ecommify_olist_analytics.sellers.")
execute_sql("CREATE INDEX IF NOT EXISTS idx_sellers_state_hash ON ecommify_olist_analytics.sellers USING HASH (seller_state);")
print("  ✅ Index 'idx_sellers_state_hash' created on ecommify_olist_analytics.sellers.")

print("\n--- Todos los índices Hash propuestos han sido creados o verificados. ---")


--- Creando índices Hash en ecommify_olist_analytics ---
  ✅ Index 'idx_customers_city_hash' created on ecommify_olist_analytics.customers.
  ✅ Index 'idx_customers_state_hash' created on ecommify_olist_analytics.customers.
  ✅ Index 'idx_orders_status_hash' created on ecommify_olist_analytics.orders.
  ✅ Index 'idx_order_payments_type_hash' created on ecommify_olist_analytics.order_payments.
  ✅ Index 'idx_sellers_city_hash' created on ecommify_olist_analytics.sellers.
  ✅ Index 'idx_sellers_state_hash' created on ecommify_olist_analytics.sellers.

--- Todos los índices Hash propuestos han sido creados o verificados. ---


### 4.3. Índice BRIN (Block Range Index)

Los índices BRIN son ideales para tablas muy grandes donde los datos están ordenados de forma natural (por ejemplo, por tiempo, ID secuencial). Son mucho más compactos que los índices B-tree y pueden ser muy eficientes para consultas de rango en estos escenarios, ya que almacenan un resumen (valores mínimo y máximo) para cada bloque de páginas físicas en el disco.

La columna `order_purchase_timestamp` en la tabla `ecommify_olist_analytics.orders` es una excelente candidata para un índice BRIN, ya que las marcas de tiempo de las compras suelen registrarse y almacenarse en orden cronológico.

In [ ]:
print("\n--- Creando índices BRIN en ecommify_olist_analytics ---")

# Indexes for 'orders' table (BRIN on purchase timestamp)
execute_sql("CREATE INDEX IF NOT EXISTS idx_orders_purchase_timestamp_brin ON ecommify_olist_analytics.orders USING BRIN (order_purchase_timestamp);")
print("  ✅ Index 'idx_orders_purchase_timestamp_brin' created on ecommify_olist_analytics.orders.")

print("\n--- Todos los índices BRIN propuestos han sido creados o verificados. ---")


--- Creando índices BRIN en ecommify_olist_analytics ---
  ✅ Index 'idx_orders_purchase_timestamp_brin' created on ecommify_olist_analytics.orders.

--- Todos los índices BRIN propuestos han sido creados o verificados. ---


## 5️. Documentación y Análisis Cuantitativo de Índices

A continuación, se documentará cada tipo de índice creado, justificando su selección, describiendo los patrones de consulta que optimiza, y analizando los _trade-offs_ considerados. Además, se medirá el impacto cuantitativo de los índices, incluyendo su tamaño y cómo modifican los planes de ejecución de consultas relevantes.

### 5.1. Índice B-tree

*   **Tipo de índice seleccionado:** B-tree
*   **Justificación técnica de selección:** Los índices B-tree son la elección predeterminada y más versátil en PostgreSQL. Son eficientes para un amplio rango de operaciones, incluyendo comparaciones de igualdad (`=`), operadores de rango (`<`, `>`, `<=`, `>=`), y ordenamiento (`ORDER BY`). Se seleccionaron para columnas que son frecuentemente utilizadas en cláusulas `WHERE`, `JOIN` y `ORDER BY`, como claves primarias, claves foráneas y columnas con alta cardinalidad donde el orden es relevante.
*   **Patrón de consulta que optimiza:**
    *   Búsquedas exactas: `SELECT * FROM customers WHERE customer_id = '...'`
    *   Búsquedas de rango: `SELECT * FROM orders WHERE order_purchase_timestamp BETWEEN '...' AND '...'`
    *   Ordenamiento: `SELECT * FROM orders ORDER BY order_purchase_timestamp DESC`
    *   Uniones (JOINs): `SELECT c.customer_id, o.order_id FROM customers c JOIN orders o ON c.customer_id = o.customer_id`
*   **Trade-offs considerados:**
    *   **Espacio vs Velocidad:** Requieren más espacio en disco que otros tipos (como BRIN) pero ofrecen un rendimiento de lectura superior para la mayoría de los patrones de consulta.
    *   **Mantenimiento:** Necesitan ser actualizados cuando los datos de la tabla cambian (INSERT, UPDATE, DELETE), lo que añade una sobrecarga en las operaciones de escritura. Sin embargo, su equilibrio entre rendimiento de lectura y costo de escritura es generalmente óptimo para cargas de trabajo transaccionales.

#### 5.1.1. Medición de Impacto Cuantitativo (B-tree)

Para evaluar el impacto, mostraremos el tamaño de algunos índices B-tree creados y los planes de ejecución de consultas representativas.

**Nota sobre Tiempo de Ejecución (Antes/Después):** Para una comparación precisa del tiempo de ejecución 'antes' y 'después', se necesitaría ejecutar consultas sin los índices (o con los índices eliminados) y luego con ellos. Dado que los índices ya están creados, solo podemos mostrar el rendimiento 'después'. El `EXPLAIN ANALYZE` nos dará una idea clara de cómo se utilizan los índices.

In [ ]:
print("\n--- Tamaño de Índices B-tree --- ")
# Obtener el tamaño de algunos índices B-tree
btree_index_sizes = execute_sql("""
SELECT
    relname AS index_name,
    pg_size_pretty(pg_relation_size(oid)) AS index_size
FROM pg_class
WHERE relkind = 'i'
AND relname IN (
    'idx_customers_unique_id',
    'idx_orders_purchase_timestamp',
    'idx_order_items_order_id'
);
""", fetch=True)

if btree_index_sizes:
    for row in btree_index_sizes:
        print(f"  ▪ {row['index_name']}: {row['index_size']}")
else:
    print("  No se pudo obtener el tamaño de los índices B-tree.")

print("\n--- Planes de Ejecución (B-tree) ---")
print("Query de ejemplo: Búsqueda por customer_unique_id en la tabla customers")
query_customers_btree = "SELECT * FROM ecommify_olist_analytics.customers WHERE customer_unique_id = '87a5a92ac19a93012921a4f00b97950c';"
explain_customers_btree = execute_sql("EXPLAIN ANALYZE " + query_customers_btree, fetch=True)
if explain_customers_btree:
    for row in explain_customers_btree:
        print(f"  {row[0]}")

print("\nQuery de ejemplo: Búsqueda de rango por order_purchase_timestamp en la tabla orders")
query_orders_btree = "SELECT order_id, customer_id, order_purchase_timestamp FROM ecommify_olist_analytics.orders WHERE order_purchase_timestamp BETWEEN '2017-01-01' AND '2017-01-31' LIMIT 10;"
explain_orders_btree = execute_sql("EXPLAIN ANALYZE " + query_orders_btree, fetch=True)
if explain_orders_btree:
    for row in explain_orders_btree:
        print(f"  {row[0]}")


--- Tamaño de Índices B-tree --- 
  ▪ idx_customers_unique_id: 5608 kB
  ▪ idx_order_items_order_id: 3256 kB
  ▪ idx_orders_purchase_timestamp: 2192 kB

--- Planes de Ejecución (B-tree) ---
Query de ejemplo: Búsqueda por customer_unique_id en la tabla customers
  Bitmap Heap Scan on customers  (cost=8.67..447.49 rows=497 width=116) (actual time=5.201..5.202 rows=0 loops=1)
    Recheck Cond: (customer_unique_id = '87a5a92ac19a93012921a4f00b97950c'::text)
    ->  Bitmap Index Scan on idx_customers_unique_id  (cost=0.00..8.54 rows=497 width=0) (actual time=4.536..4.537 rows=0 loops=1)
          Index Cond: (customer_unique_id = '87a5a92ac19a93012921a4f00b97950c'::text)
  Planning Time: 2.237 ms
  Execution Time: 5.239 ms

Query de ejemplo: Búsqueda de rango por order_purchase_timestamp en la tabla orders
  Limit  (cost=0.29..10.75 rows=10 width=40) (actual time=0.011..0.026 rows=10 loops=1)
    ->  Index Scan using idx_orders_purchase_timestamp on orders  (cost=0.29..232.43 rows=222 widt

---

### 5.2. Índice Hash

*   **Tipo de índice seleccionado:** Hash
*   **Justificación técnica de selección:** Los índices Hash están optimizados para búsquedas de igualdad exacta (`=`). Son particularmente útiles para columnas con muchos valores distintos pero donde las consultas se centran exclusivamente en encontrar un valor específico. Se seleccionaron para columnas categóricas o de identificadores que no requieren ordenamiento ni búsquedas de rango, como `customer_city` o `order_status`.
*   **Patrón de consulta que optimiza:** Búsquedas exactas: `SELECT * FROM customers WHERE customer_city = 'sao paulo'`
*   **Trade-offs considerados:**
    *   **Espacio vs Velocidad:** Generalmente son más compactos que los B-tree para búsquedas de igualdad exacta, lo que puede resultar en una ligera ventaja de velocidad en ese escenario. Sin embargo, no soportan búsquedas de rango ni ordenamiento.
    *   **Mantenimiento:** Similar a los B-tree, requieren actualización con cambios en los datos. Históricamente, los índices Hash de PostgreSQL eran menos robustos y no eran replicados; sin embargo, las versiones modernas han mejorado esto, aunque los B-tree siguen siendo la opción por defecto debido a su mayor versatilidad.

#### 5.2.1. Medición de Impacto Cuantitativo (Hash)

In [ ]:
print("\n--- Tamaño de Índices Hash --- ")
# Obtener el tamaño de algunos índices Hash
hash_index_sizes = execute_sql("""
SELECT
    relname AS index_name,
    pg_size_pretty(pg_relation_size(oid)) AS index_size
FROM pg_class
WHERE relkind = 'i'
AND relname IN (
    'idx_customers_city_hash',
    'idx_orders_status_hash'
);
""", fetch=True)

if hash_index_sizes:
    for row in hash_index_sizes:
        print(f"  ▪ {row['index_name']}: {row['index_size']}")
else:
    print("  No se pudo obtener el tamaño de los índices Hash.")

print("\n--- Planes de Ejecución (Hash) ---")
print("Query de ejemplo: Búsqueda exacta por customer_city en la tabla customers")
query_customers_hash = "SELECT * FROM ecommify_olist_analytics.customers WHERE customer_city = 'sao paulo' LIMIT 10;"
explain_customers_hash = execute_sql("EXPLAIN ANALYZE " + query_customers_hash, fetch=True)
if explain_customers_hash:
    for row in explain_customers_hash:
        print(f"  {row[0]}")


--- Tamaño de Índices Hash --- 
  ▪ idx_customers_city_hash: 5040 kB
  ▪ idx_orders_status_hash: 6040 kB

--- Planes de Ejecución (Hash) ---
Query de ejemplo: Búsqueda exacta por customer_city en la tabla customers
  Limit  (cost=0.00..9.45 rows=10 width=116) (actual time=0.024..0.030 rows=10 loops=1)
    ->  Index Scan using idx_customers_city_hash on customers  (cost=0.00..469.60 rows=497 width=116) (actual time=0.023..0.028 rows=10 loops=1)
          Index Cond: (customer_city = 'sao paulo'::text)
  Planning Time: 0.114 ms
  Execution Time: 0.052 ms


---

### 5.3. Índice BRIN (Block Range Index)

*   **Tipo de índice seleccionado:** BRIN
*   **Justificación técnica de selección:** Los índices BRIN son ideales para tablas muy grandes donde los datos están naturalmente ordenados en disco. Son muy compactos y eficientes para consultas de rango en estos escenarios. Se seleccionó para la columna `order_purchase_timestamp` en la tabla `orders` porque las marcas de tiempo de las compras se insertan típicamente de forma secuencial, lo que resulta en un almacenamiento físicamente ordenado de los datos. Esto permite al índice BRIN almacenar un resumen de los rangos de valores para cada bloque de páginas, evitando escanear bloques irrelevantes.
*   **Patrón de consulta que optimiza:** Búsquedas de rango: `SELECT * FROM orders WHERE order_purchase_timestamp BETWEEN '2018-01-01' AND '2018-03-31'`
*   **Trade-offs considerados:**
    *   **Espacio vs Velocidad:** Los índices BRIN son extremadamente compactos y ocupan mucho menos espacio que los B-tree. Su rendimiento para consultas de rango en datos ordenados es excelente, aunque no son adecuados para búsquedas exactas o rangos en datos desordenados.
    *   **Mantenimiento:** Tienen un costo de mantenimiento muy bajo, ya que solo necesitan actualizar sus resúmenes de rango de vez en cuando. Esto los hace ideales para tablas de hechos o registros de log que crecen constantemente de manera ordenada.

#### 5.3.1. Medición de Impacto Cuantitativo (BRIN)

In [ ]:
print("\n--- Tamaño de Índices BRIN --- ")
# Obtener el tamaño de los índices BRIN
brin_index_sizes = execute_sql("""
SELECT
    relname AS index_name,
    pg_size_pretty(pg_relation_size(oid)) AS index_size
FROM pg_class
WHERE relkind = 'i'
AND relname IN (
    'idx_orders_purchase_timestamp_brin'
);
""", fetch=True)

if brin_index_sizes:
    for row in brin_index_sizes:
        print(f"  ▪ {row['index_name']}: {row['index_size']}")
else:
    print("  No se pudo obtener el tamaño de los índices BRIN.")

print("\n--- Planes de Ejecución (BRIN) ---")
print("Query de ejemplo: Búsqueda de rango en order_purchase_timestamp")
query_orders_brin = "SELECT order_id, order_purchase_timestamp FROM ecommify_olist_analytics.orders WHERE order_purchase_timestamp BETWEEN '2018-01-01' AND '2018-03-31' LIMIT 10;"
explain_orders_brin = execute_sql("EXPLAIN ANALYZE " + query_orders_brin, fetch=True)
if explain_orders_brin:
    for row in explain_orders_brin:
        print(f"  {row[0]}")


--- Tamaño de Índices BRIN --- 
  ▪ idx_orders_purchase_timestamp_brin: 24 kB

--- Planes de Ejecución (BRIN) ---
Query de ejemplo: Búsqueda de rango en order_purchase_timestamp
  Limit  (cost=0.29..1.26 rows=10 width=24) (actual time=1.852..1.869 rows=10 loops=1)
    ->  Index Scan using idx_orders_purchase_timestamp on orders  (cost=0.29..2048.66 rows=21129 width=24) (actual time=1.850..1.867 rows=10 loops=1)
          Index Cond: ((order_purchase_timestamp >= '2018-01-01 00:00:00'::timestamp without time zone) AND (order_purchase_timestamp <= '2018-03-31 00:00:00'::timestamp without time zone))
  Planning Time: 0.121 ms
  Execution Time: 1.899 ms


## 6️. Particionamiento Declarativo

### 6.1. Análisis y Selección de Tablas Candidatas

Para realizar el particionamiento declarativo, primero identificaremos las tablas dentro del esquema `ecommify_olist_analytics` que contienen más de 100,000 registros. Estas tablas son las principales candidatas para beneficiarse de esta técnica de optimización, ya que el particionamiento ayuda a gestionar grandes volúmenes de datos de manera más eficiente.

In [ ]:
from psycopg2 import sql # Added to ensure sql object is defined

print("--- Contar registros en las tablas del esquema 'ecommify_olist_analytics' ---")
tables_to_check = [
    'customers',
    'orders',
    'order_items',
    'order_payments',
    'sellers'
]

for table_name in tables_to_check:
    query = sql.SQL("SELECT COUNT(*) FROM ecommify_olist_analytics.{}; ").format(sql.Identifier(table_name))
    count_result = execute_sql(query.as_string(get_connection().cursor()), fetch=True)
    if count_result:
        row_count = count_result[0][0] # Access the count value from the DictRow
        print(f"  Tabla '{table_name}': {row_count} registros")
    else:
        print(f"  No se pudo obtener el conteo de registros para la tabla '{table_name}'.")

--- Contar registros en las tablas del esquema 'ecommify_olist_analytics' ---
  Tabla 'customers': 99441 registros
  Tabla 'orders': 99441 registros
  Tabla 'order_items': 112650 registros
  Tabla 'order_payments': 103886 registros
  Tabla 'sellers': 3095 registros


### 6.2. Análisis de Patrones de Consulta y Selección de Columna de Partición

Para seleccionar la columna de partición más adecuada, es crucial entender los tipos de datos disponibles y los patrones de consulta comunes. La partición por `RANGE` es ideal para columnas con un orden natural, como fechas, marcas de tiempo o identificadores secuenciales. A continuación, obtendremos el esquema de las tablas candidatas (`order_items` y `order_payments`) para identificar posibles columnas de partición.

In [ ]:
print("--- Esquema de la tabla 'order_items' ---")
schema_order_items = execute_sql(
    """SELECT column_name, data_type FROM information_schema.columns WHERE table_schema = 'ecommify_olist_analytics' AND table_name = 'order_items';""",
    fetch=True
)
if schema_order_items:
    for col in schema_order_items:
        print(f"  ▪ {col['column_name']}: {col['data_type']}")
else:
    print("  No se pudo obtener el esquema de 'order_items'.")

print("\n--- Esquema de la tabla 'order_payments' ---")
schema_order_payments = execute_sql(
    """SELECT column_name, data_type FROM information_schema.columns WHERE table_schema = 'ecommify_olist_analytics' AND table_name = 'order_payments';""",
    fetch=True
)
if schema_order_payments:
    for col in schema_order_payments:
        print(f"  ▪ {col['column_name']}: {col['data_type']}")
else:
    print("  No se pudo obtener el esquema de 'order_payments'.")

--- Esquema de la tabla 'order_items' ---
  ▪ order_id: uuid
  ▪ order_item_id: integer
  ▪ product_id: uuid
  ▪ seller_id: uuid
  ▪ shipping_limit_date: timestamp without time zone
  ▪ price: numeric
  ▪ freight_value: numeric

--- Esquema de la tabla 'order_payments' ---
  ▪ order_id: uuid
  ▪ payment_sequential: numeric
  ▪ payment_type: text
  ▪ payment_installments: numeric
  ▪ payment_value: numeric


### 6.2.1. Determinando Rangos de Partición

Antes de crear las particiones por rango, es esencial conocer el mínimo y máximo de las columnas seleccionadas (`shipping_limit_date` para `order_items` y `payment_value` para `order_payments`). Esto nos permitirá definir los límites de cada partición de manera lógica y eficiente.

In [ ]:
print("--- Rango de Fechas para 'shipping_limit_date' en order_items ---")
shipping_dates_range = execute_sql(
    "SELECT MIN(shipping_limit_date), MAX(shipping_limit_date) FROM ecommify_olist_analytics.order_items;",
    fetch=True
)

if shipping_dates_range:
    min_date = shipping_dates_range[0][0]
    max_date = shipping_dates_range[0][1]
    print(f"  Fecha mínima de envío: {min_date}")
    print(f"  Fecha máxima de envío: {max_date}")
else:
    print("  No se pudo obtener el rango de fechas para 'order_items'.")

print("\n--- Rango de Valores para 'payment_value' en order_payments ---")
payment_values_range = execute_sql(
    "SELECT MIN(payment_value), MAX(payment_value) FROM ecommify_olist_analytics.order_payments;",
    fetch=True
)

if payment_values_range:
    min_value = payment_values_range[0][0]
    max_value = payment_values_range[0][1]
    print(f"  Valor de pago mínimo: {min_value}")
    print(f"  Valor de pago máximo: {max_value}")
else:
    print("  No se pudo obtener el rango de valores para 'order_payments'.")

--- Rango de Fechas para 'shipping_limit_date' en order_items ---
  Fecha mínima de envío: 2016-09-19 00:15:34
  Fecha máxima de envío: 2020-04-09 22:35:08

--- Rango de Valores para 'payment_value' en order_payments ---
  Valor de pago mínimo: 0.0
  Valor de pago máximo: 13664.08


### 6.2.2. Creación de Tablas Particionadas

Procederemos a crear las tablas particionadas `order_items_partitioned` y `order_payments_partitioned` en el esquema `ecommify_olist_analytics`. Estas nuevas tablas reemplazarán las originales en términos de funcionalidad de consulta optimizada. Luego, se crearán las particiones hijas y se migrarán los datos.

In [ ]:
# --- Particionamiento para order_items por shipping_limit_date ---
print("\n--- Creando tabla particionada para 'order_items' ---")

# Eliminar la tabla particionada si ya existe (y sus hijos)
execute_sql("DROP TABLE IF EXISTS ecommify_olist_analytics.order_items_partitioned CASCADE;")
print("  ✅ Tabla 'order_items_partitioned' (y sus hijos) eliminada si existía.")

# Crear la tabla padre particionada
create_partitioned_items_query = """
CREATE TABLE ecommify_olist_analytics.order_items_partitioned (
    order_id uuid,
    order_item_id integer,
    product_id uuid,
    seller_id uuid,
    shipping_limit_date timestamp without time zone,
    price numeric,
    freight_value numeric
) PARTITION BY RANGE (shipping_limit_date);
"""
execute_sql(create_partitioned_items_query)
print("  ✅ Tabla padre 'order_items_partitioned' creada.")

# Crear particiones hijas para order_items
print("\n  Creando particiones hijas para 'order_items_partitioned'...")
execute_sql("CREATE TABLE ecommify_olist_analytics.order_items_p2016 PARTITION OF ecommify_olist_analytics.order_items_partitioned FOR VALUES FROM ('2016-01-01') TO ('2017-01-01');")
print("    ▪ Partición 'order_items_p2016' creada.")
execute_sql("CREATE TABLE ecommify_olist_analytics.order_items_p2017 PARTITION OF ecommify_olist_analytics.order_items_partitioned FOR VALUES FROM ('2017-01-01') TO ('2018-01-01');")
print("    ▪ Partición 'order_items_p2017' creada.")
execute_sql("CREATE TABLE ecommify_olist_analytics.order_items_p2018 PARTITION OF ecommify_olist_analytics.order_items_partitioned FOR VALUES FROM ('2018-01-01') TO ('2019-01-01');")
print("    ▪ Partición 'order_items_p2018' creada.")
execute_sql("CREATE TABLE ecommify_olist_analytics.order_items_p2019 PARTITION OF ecommify_olist_analytics.order_items_partitioned FOR VALUES FROM ('2019-01-01') TO ('2020-01-01');")
print("    ▪ Partición 'order_items_p2019' creada.")
execute_sql("CREATE TABLE ecommify_olist_analytics.order_items_p2020 PARTITION OF ecommify_olist_analytics.order_items_partitioned FOR VALUES FROM ('2020-01-01') TO ('2021-01-01');")
print("    ▪ Partición 'order_items_p2020' creada.")

print("\n--- Migrando datos de 'order_items' a 'order_items_partitioned' ---")
execute_sql("INSERT INTO ecommify_olist_analytics.order_items_partitioned SELECT * FROM ecommify_olist_analytics.order_items;")
print("  ✅ Datos migrados a 'order_items_partitioned'.")


# --- Particionamiento para order_payments por payment_value ---
print("\n--- Creando tabla particionada para 'order_payments' ---")

# Eliminar la tabla particionada si ya existe (y sus hijos)
execute_sql("DROP TABLE IF EXISTS ecommify_olist_analytics.order_payments_partitioned CASCADE;")
print("  ✅ Tabla 'order_payments_partitioned' (y sus hijos) eliminada si existía.")

# Crear la tabla padre particionada
create_partitioned_payments_query = """
CREATE TABLE ecommify_olist_analytics.order_payments_partitioned (
    order_id uuid,
    payment_sequential numeric,
    payment_type text,
    payment_installments numeric,
    payment_value numeric
) PARTITION BY RANGE (payment_value);
"""
execute_sql(create_partitioned_payments_query)
print("  ✅ Tabla padre 'order_payments_partitioned' creada.")

# Crear particiones hijas para order_payments
print("\n  Creando particiones hijas para 'order_payments_partitioned'...")
execute_sql("CREATE TABLE ecommify_olist_analytics.order_payments_p_less_10 PARTITION OF ecommify_olist_analytics.order_payments_partitioned FOR VALUES FROM (0.0) TO (10.0);")
print("    ▪ Partición 'order_payments_p_less_10' creada.")
execute_sql("CREATE TABLE ecommify_olist_analytics.order_payments_p_10_to_100 PARTITION OF ecommify_olist_analytics.order_payments_partitioned FOR VALUES FROM (10.0) TO (100.0);")
print("    ▪ Partición 'order_payments_p_10_to_100' creada.")
execute_sql("CREATE TABLE ecommify_olist_analytics.order_payments_p_100_to_500 PARTITION OF ecommify_olist_analytics.order_payments_partitioned FOR VALUES FROM (100.0) TO (500.0);")
print("    ▪ Partición 'order_payments_p_100_to_500' creada.")
execute_sql("CREATE TABLE ecommify_olist_analytics.order_payments_p_500_to_1000 PARTITION OF ecommify_olist_analytics.order_payments_partitioned FOR VALUES FROM (500.0) TO (1000.0);")
print("    ▪ Partición 'order_payments_p_500_to_1000' creada.")
execute_sql("CREATE TABLE ecommify_olist_analytics.order_payments_p_1000_plus PARTITION OF ecommify_olist_analytics.order_payments_partitioned FOR VALUES FROM (1000.0) TO (MAXVALUE);")
print("    ▪ Partición 'order_payments_p_1000_plus' creada.")

print("\n--- Migrando datos de 'order_payments' a 'order_payments_partitioned' ---")
execute_sql("INSERT INTO ecommify_olist_analytics.order_payments_partitioned SELECT * FROM ecommify_olist_analytics.order_payments;")
print("  ✅ Datos migrados a 'order_payments_partitioned'.")


--- Creando tabla particionada para 'order_items' ---
  ✅ Tabla 'order_items_partitioned' (y sus hijos) eliminada si existía.
  ✅ Tabla padre 'order_items_partitioned' creada.

  Creando particiones hijas para 'order_items_partitioned'...
    ▪ Partición 'order_items_p2016' creada.
    ▪ Partición 'order_items_p2017' creada.
    ▪ Partición 'order_items_p2018' creada.
    ▪ Partición 'order_items_p2019' creada.
    ▪ Partición 'order_items_p2020' creada.

--- Migrando datos de 'order_items' a 'order_items_partitioned' ---
  ✅ Datos migrados a 'order_items_partitioned'.

--- Creando tabla particionada para 'order_payments' ---
  ✅ Tabla 'order_payments_partitioned' (y sus hijos) eliminada si existía.
  ✅ Tabla padre 'order_payments_partitioned' creada.

  Creando particiones hijas para 'order_payments_partitioned'...
    ▪ Partición 'order_payments_p_less_10' creada.
    ▪ Partición 'order_payments_p_10_to_100' creada.
    ▪ Partición 'order_payments_p_100_to_500' creada.
    ▪ Partici

### 6.3. Particionamiento por LISTA (LIST Partitioning)

El particionamiento por `LIST` es útil cuando se desea dividir una tabla basándose en valores discretos de una columna. Cada partición se define explícitamente para un conjunto específico de valores. Esto es ideal para columnas categóricas como `payment_type`, `order_status`, o códigos de región, donde las consultas a menudo filtran por uno o varios de estos valores conocidos.

### 6.3.1. Análisis y Selección de Columna de Partición para LIST

Para el particionamiento por `LIST` en `order_payments`, la columna `payment_type` es una excelente candidata, ya que contiene un número manejable de valores discretos. Necesitamos obtener estos valores únicos para definir nuestras particiones.

In [ ]:
print("\n--- Obteniendo valores únicos de 'payment_type' para particionamiento LIST ---")
payment_types = execute_sql(
    "SELECT DISTINCT payment_type FROM ecommify_olist_analytics.order_payments ORDER BY payment_type;",
    fetch=True
)

if payment_types:
    print("  Valores únicos de payment_type:")
    for pt in payment_types:
        print(f"    ▪ {pt['payment_type']}")
    unique_payment_types = [pt['payment_type'] for pt in payment_types]
else:
    print("  No se pudieron obtener los tipos de pago.")
    unique_payment_types = []


--- Obteniendo valores únicos de 'payment_type' para particionamiento LIST ---
  Valores únicos de payment_type:
    ▪ boleto
    ▪ credit_card
    ▪ debit_card
    ▪ not_defined
    ▪ voucher


### 6.3.2. Creación de Tabla Particionada por LISTA

Procederemos a crear la tabla particionada `order_payments_partitioned_list` en el esquema `ecommify_olist_analytics`. Se eliminará la tabla `order_payments_partitioned` previamente creada con `RANGE` para evitar conflictos y se creará la nueva con `LIST` partición. Luego, se crearán las particiones hijas y se migrarán los datos.

In [ ]:
# --- Particionamiento para order_payments por payment_type (LIST) ---
print("\n--- Creando tabla particionada para 'order_payments' por LIST ---")

# Eliminar la tabla particionada por RANGE si ya existe (y sus hijos)
execute_sql("DROP TABLE IF EXISTS ecommify_olist_analytics.order_payments_partitioned CASCADE;")
print("  ✅ Tabla 'order_payments_partitioned' (y sus hijos) eliminada si existía.")

# Eliminar la tabla particionada por LIST si ya existe (y sus hijos)
execute_sql("DROP TABLE IF EXISTS ecommify_olist_analytics.order_payments_partitioned_list CASCADE;")
print("  ✅ Tabla 'order_payments_partitioned_list' (y sus hijos) eliminada si existía.")

# Crear la tabla padre particionada por LIST
create_partitioned_payments_list_query = """
CREATE TABLE ecommify_olist_analytics.order_payments_partitioned_list (
    order_id uuid,
    payment_sequential numeric,
    payment_type text,
    payment_installments numeric,
    payment_value numeric
) PARTITION BY LIST (payment_type);
"""
execute_sql(create_partitioned_payments_list_query)
print("  ✅ Tabla padre 'order_payments_partitioned_list' creada.")

# Crear particiones hijas para order_payments por LIST
print("\n  Creando particiones hijas para 'order_payments_partitioned_list'...")
for pt_value in unique_payment_types:
    partition_name = f"order_payments_p_{pt_value.replace('-', '_').lower()}"
    create_partition_query = sql.SQL("CREATE TABLE ecommify_olist_analytics.{} PARTITION OF ecommify_olist_analytics.order_payments_partitioned_list FOR VALUES IN ({});").format(
        sql.Identifier(partition_name),
        sql.Literal(pt_value)
    ).as_string(get_connection().cursor())
    execute_sql(create_partition_query)
    print(f"    ▪ Partición '{partition_name}' creada para '{pt_value}'.")

print("\n--- Migrando datos de 'order_payments' a 'order_payments_partitioned_list' ---")
execute_sql("INSERT INTO ecommify_olist_analytics.order_payments_partitioned_list SELECT * FROM ecommify_olist_analytics.order_payments;")
print("  ✅ Datos migrados a 'order_payments_partitioned_list'.")


--- Creando tabla particionada para 'order_payments' por LIST ---
  ✅ Tabla 'order_payments_partitioned' (y sus hijos) eliminada si existía.
  ✅ Tabla 'order_payments_partitioned_list' (y sus hijos) eliminada si existía.
  ✅ Tabla padre 'order_payments_partitioned_list' creada.

  Creando particiones hijas para 'order_payments_partitioned_list'...
    ▪ Partición 'order_payments_p_boleto' creada para 'boleto'.
    ▪ Partición 'order_payments_p_credit_card' creada para 'credit_card'.
    ▪ Partición 'order_payments_p_debit_card' creada para 'debit_card'.
    ▪ Partición 'order_payments_p_not_defined' creada para 'not_defined'.
    ▪ Partición 'order_payments_p_voucher' creada para 'voucher'.

--- Migrando datos de 'order_payments' a 'order_payments_partitioned_list' ---
  ✅ Datos migrados a 'order_payments_partitioned_list'.


### 6.4. Documentación y Análisis Cuantitativo del Particionamiento por LISTA

*   **Tipo de Particionamiento seleccionado:** LIST
*   **Justificación técnica de selección:** El particionamiento por `LIST` es ideal para la columna `payment_type` ya que esta columna contiene un conjunto finito y discreto de valores. Permite que las consultas que filtran por `payment_type` accedan solo a la partición relevante, lo que reduce la cantidad de datos que el motor de la base de datos necesita escanear.
*   **Patrón de consulta que optimiza:**
    *   Búsquedas exactas por tipo de pago: `SELECT * FROM ecommify_olist_analytics.order_payments_partitioned_list WHERE payment_type = 'credit_card'`
    *   Agregaciones por tipo de pago: `SELECT payment_type, COUNT(*) FROM ecommify_olist_analytics.order_payments_partitioned_list GROUP BY payment_type`
*   **Trade-offs considerados:**
    *   **Gestión de Particiones:** Requiere que se definan explícitamente todos los valores posibles para la partición. Si un nuevo `payment_type` aparece en los datos, se debe crear una nueva partición antes de que los datos puedan ser insertados, o usar una partición `DEFAULT`.
    *   **Rendimiento:** Mejora significativamente el rendimiento para consultas que involucran la columna de partición, ya que el planificador de consultas puede eliminar automáticamente las particiones que no contienen los datos buscados (exclusión de particiones).
    *   **Complejidad:** Añade una capa de complejidad en la administración de la base de datos en comparación con una tabla no particionada, especialmente cuando se manejan muchos valores discretos o cuando los valores pueden cambiar con el tiempo.

#### 6.4.1. Medición de Impacto Cuantitativo (Particionamiento LIST)

Mostraremos el tamaño de algunas particiones de `order_payments_partitioned_list` y los planes de ejecución de consultas representativas para evaluar el impacto del particionamiento por `LIST`.

In [ ]:
print("\n--- Tamaño de Particiones (LIST) para 'order_payments_partitioned_list' ---")
list_partition_sizes = execute_sql("""
SELECT
    relname AS partition_name,
    pg_size_pretty(pg_relation_size(oid)) AS partition_size
FROM pg_class
WHERE relkind = 'r' -- 'r' for regular table (which partitions are)
AND relnamespace = (SELECT oid FROM pg_namespace WHERE nspname = 'ecommify_olist_analytics')
AND relname LIKE 'order_payments_p_%'
ORDER BY relname;
""", fetch=True)

if list_partition_sizes:
    for row in list_partition_sizes:
        print(f"  ▪ {row['partition_name']}: {row['partition_size']}")
else:
    print("  No se pudo obtener el tamaño de las particiones LIST.")

print("\n--- Planes de Ejecución (Particionamiento LIST) ---")
print("Query de ejemplo: Búsqueda por 'payment_type' = 'credit_card'")
query_list_partition = "SELECT * FROM ecommify_olist_analytics.order_payments_partitioned_list WHERE payment_type = 'credit_card' LIMIT 10;"
explain_list_partition = execute_sql("EXPLAIN ANALYZE " + query_list_partition, fetch=True)
if explain_list_partition:
    for row in explain_list_partition:
        print(f"  {row[0]}")

print("\nQuery de ejemplo: Búsqueda por 'payment_type' = 'boleto' (otra partición)")
query_list_partition_2 = "SELECT COUNT(*) FROM ecommify_olist_analytics.order_payments_partitioned_list WHERE payment_type = 'boleto';"
explain_list_partition_2 = execute_sql("EXPLAIN ANALYZE " + query_list_partition_2, fetch=True)
if explain_list_partition_2:
    for row in explain_list_partition_2:
        print(f"  {row[0]}")


--- Tamaño de Particiones (LIST) para 'order_payments_partitioned_list' ---
  ▪ order_payments_p_boleto: 1320 kB
  ▪ order_payments_p_credit_card: 5744 kB
  ▪ order_payments_p_debit_card: 120 kB
  ▪ order_payments_p_not_defined: 8192 bytes
  ▪ order_payments_p_voucher: 424 kB

--- Planes de Ejecución (Particionamiento LIST) ---
Query de ejemplo: Búsqueda por 'payment_type' = 'credit_card'
  Limit  (cost=0.00..67.45 rows=10 width=144) (actual time=0.021..0.024 rows=10 loops=1)
    ->  Seq Scan on order_payments_p_credit_card order_payments_partitioned_list  (cost=0.00..1139.83 rows=169 width=144) (actual time=0.020..0.021 rows=10 loops=1)
          Filter: (payment_type = 'credit_card'::text)
  Planning Time: 0.137 ms
  Execution Time: 0.061 ms

Query de ejemplo: Búsqueda por 'payment_type' = 'boleto' (otra partición)
  Aggregate  (cost=262.04..262.05 rows=1 width=8) (actual time=5.223..5.224 rows=1 loops=1)
    ->  Seq Scan on order_payments_p_boleto order_payments_partitioned_list  (

## 7️. Comparación de Rendimiento: Esquemas `public` vs. `ecommify_olist_analytics`

Para validar el impacto de la indexación y el particionamiento, realizaremos una comparación directa de los planes de ejecución y los tiempos de consulta entre las tablas originales del esquema `public` y las tablas optimizadas (con índices y particiones) del esquema `ecommify_olist_analytics`. Esto nos permitirá cuantificar las mejoras de rendimiento logradas.

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Helper function to run EXPLAIN ANALYZE and extract execution time
def run_explain_and_get_time(query):
    explain_result = execute_sql("EXPLAIN ANALYZE " + query, fetch=True)
    if explain_result:
        full_output = "\n".join([row[0] for row in explain_result])
        # Find the line with 'Execution Time:' (it's usually the last one)
        for line in reversed(explain_result):
            if 'Execution Time:' in line[0]:
                try:
                    time_str = line[0].split('Execution Time: ')[1].split(' ')[0]
                    return float(time_str), full_output
                except (IndexError, ValueError):
                    pass
        return None, full_output # Return None if time not found but output exists
    return None, "No EXPLAIN ANALYZE output."


# Define the queries to compare
queries_to_compare = [
    {
        "name": "Customers - Unique ID (B-tree)",
        "query_template": "SELECT * FROM {schema}.customers WHERE customer_unique_id = '87a5a92ac19a93012921a4f00b97950c';",
        "table_public": "public.customers",
        "table_analytics": "ecommify_olist_analytics.customers"
    },
    {
        "name": "Orders - Purchase Timestamp Range (B-tree/BRIN)",
        "query_template": "SELECT order_id, customer_id, order_purchase_timestamp FROM {schema}.orders WHERE order_purchase_timestamp BETWEEN '2017-01-01' AND '2017-01-31' LIMIT 10;",
        "table_public": "public.orders",
        "table_analytics": "ecommify_olist_analytics.orders"
    },
    {
        "name": "Order Items - Shipping Date Range (Partitioned RANGE)",
        "query_template_public": "SELECT * FROM public.order_items WHERE shipping_limit_date BETWEEN '2017-01-01' AND '2017-01-31' LIMIT 10;",
        "query_template_analytics": "SELECT * FROM ecommify_olist_analytics.order_items_partitioned WHERE shipping_limit_date BETWEEN '2017-01-01' AND '2017-01-31' LIMIT 10;",
        "table_public": "public.order_items",
        "table_analytics": "ecommify_olist_analytics.order_items_partitioned"
    },
    {
        "name": "Order Payments - Payment Type (Partitioned LIST)",
        "query_template_public": "SELECT * FROM public.order_payments WHERE payment_type = 'credit_card' LIMIT 10;",
        "query_template_analytics": "SELECT * FROM ecommify_olist_analytics.order_payments_partitioned_list WHERE payment_type = 'credit_card' LIMIT 10;",
        "table_public": "public.order_payments",
        "table_analytics": "ecommify_olist_analytics.order_payments_partitioned_list"
    },
    {
        "name": "Sellers - Zip Code (B-tree)",
        "query_template": "SELECT * FROM {schema}.sellers WHERE seller_zip_code_prefix = '13028';",
        "table_public": "public.sellers",
        "table_analytics": "ecommify_olist_analytics.sellers"
    }
]

comparison_results = []
explain_outputs = {}

print("\n--- Ejecutando EXPLAIN ANALYZE para consultas comparativas ---")
for q_info in queries_to_compare:
    query_name = q_info["name"]

    # Public Schema
    if "query_template_public" in q_info:
        public_query = q_info["query_template_public"]
    elif "query_template" in q_info:
        public_query = q_info["query_template"].format(schema="public")
    else:
        public_query = ""
        print(f"  ⚠️ Advertencia: No se encontró plantilla de consulta para el esquema público en: {query_name}")

    print(f"\n  - Analizando: {query_name} (Esquema Public)")
    if public_query:
        print(f"    Consulta Public: {public_query}") # Print the query
        public_time, public_explain = run_explain_and_get_time(public_query)
        explain_outputs[f"Public - {query_name}"] = public_explain
        comparison_results.append({
            "Query": query_name,
            "Schema": "public",
            "Execution Time (ms)": public_time
        })
    else:
        comparison_results.append({
            "Query": query_name,
            "Schema": "public",
            "Execution Time (ms)": None # Or a suitable indicator for skipped queries
        })

    # Analytics Schema
    if "query_template_analytics" in q_info:
        analytics_query = q_info["query_template_analytics"]
    elif "query_template" in q_info:
        analytics_query = q_info["query_template"].format(schema="ecommify_olist_analytics")
    else:
        analytics_query = ""
        print(f"  ⚠️ Advertencia: No se encontró plantilla de consulta para el esquema analytics en: {query_name}")

    print(f"  - Analizando: {query_name} (Esquema Analytics)")
    if analytics_query:
        print(f"    Consulta Analytics: {analytics_query}") # Print the query
        analytics_time, analytics_explain = run_explain_and_get_time(analytics_query)
        explain_outputs[f"Analytics - {query_name}"] = analytics_explain
        comparison_results.append({
            "Query": query_name,
            "Schema": "ecommify_olist_analytics",
            "Execution Time (ms)": analytics_time
        })
    else:
        comparison_results.append({
            "Query": query_name,
            "Schema": "ecommify_olist_analytics",
            "Execution Time (ms)": None # Or a suitable indicator for skipped queries
        })

# Convert results to DataFrame
df_results = pd.DataFrame(comparison_results)

print("\n--- Resultados EXPLAIN ANALYZE ---")
for key, value in explain_outputs.items():
    print(f"\n### {key} ###")
    print(value)

print("\nDataFrame de resultados de tiempo de ejecución:")
display(df_results)


--- Ejecutando EXPLAIN ANALYZE para consultas comparativas ---

  - Analizando: Customers - Unique ID (B-tree) (Esquema Public)
    Consulta Public: SELECT * FROM public.customers WHERE customer_unique_id = '87a5a92ac19a93012921a4f00b97950c';
  - Analizando: Customers - Unique ID (B-tree) (Esquema Analytics)
    Consulta Analytics: SELECT * FROM ecommify_olist_analytics.customers WHERE customer_unique_id = '87a5a92ac19a93012921a4f00b97950c';

  - Analizando: Orders - Purchase Timestamp Range (B-tree/BRIN) (Esquema Public)
    Consulta Public: SELECT order_id, customer_id, order_purchase_timestamp FROM public.orders WHERE order_purchase_timestamp BETWEEN '2017-01-01' AND '2017-01-31' LIMIT 10;
  - Analizando: Orders - Purchase Timestamp Range (B-tree/BRIN) (Esquema Analytics)
    Consulta Analytics: SELECT order_id, customer_id, order_purchase_timestamp FROM ecommify_olist_analytics.orders WHERE order_purchase_timestamp BETWEEN '2017-01-01' AND '2017-01-31' LIMIT 10;

  - Analizando: O

,Query,Schema,Execution Time (ms)
0,Customers - Unique ID (B-tree),public,18.171
1,Customers - Unique ID (B-tree),ecommify_olist_analytics,0.110
2,Orders - Purchase Timestamp Range (B-tree/BRIN),public,0.468
3,Orders - Purchase Timestamp Range (B-tree/BRIN),ecommify_olist_analytics,0.074
4,Order Items - Shipping Date Range (Partitioned...,public,10.011
5,Order Items - Shipping Date Range (Partitioned...,ecommify_olist_analytics,1.890
6,Order Payments - Payment Type (Partitioned LIST),public,0.636
7,Order Payments - Payment Type (Partitioned LIST),ecommify_olist_analytics,1.738
8,Sellers - Zip Code (B-tree),public,35.065
9,Sellers - Zip Code (B-tree),ecommify_olist_analytics,3.883


### Visualización de Tiempos de Ejecución

In [ ]:
import plotly.express as px

# Plotting with Plotly - Horizontal Bar Chart
fig = px.bar(df_results, y="Query", x="Execution Time (ms)", color="Schema",
             barmode="group", orientation='h', # Set orientation to 'h' for horizontal bars
             title="Comparación de Tiempos de Ejecución de Consultas (Public vs. Analytics)",
             height=700) # Adjust height as needed

fig.update_layout(yaxis_title="Tipo de Consulta", xaxis_title="Tiempo de Ejecución (ms)",
                  legend_title="Esquema",
                  yaxis={'categoryorder':'total ascending'}) # Order queries for consistency

fig.show()


### Resumen de Mejoras de Rendimiento (Gráfico Cuantitativo)